In [ ]:
!pip -q install datasets

In [ ]:
%%writefile translit_uz.py
"""
translit_uz.py -- rule-based Uzbek Latin <-> Cyrillic transliteration.

Design goals
------------
* Deterministic and fully documented (so the paper can state the exact rules).
* Latin -> Cyrillic is the direction used in the experiments.
* Cyrillic -> Latin is provided for round-trip diagnostics only.

KNOWN, UNAVOIDABLE LIMITATIONS (must be reported in the paper)
--------------------------------------------------------------
1. Russian loanwords: Cyrillic Uzbek keeps etymological letters (ц, щ, ь, ы, ъ in
   some loans, ё, ю, я in loans) that the Latin script has already merged
   (e.g. s for both с and ц). A Latin->Cyrillic mapping therefore CANNOT recover them.
   A small stem list (TS_STEMS) restores ц in the commonest -tsiya/-tsion words;
   everything else is rendered with the regular mapping.
2. 'sh', 'ch', 'ng' digraphs are treated as single units; the rare sequences s+h,
   c+h, n+g (e.g. 'mashhur' is fine, 'ongli' is fine) are not disambiguated.
3. 'ye/yo/yu/ya' -> е/ё/ю/я, 'y' + other -> й.
4. 'e' -> э at word start or after a vowel, otherwise е.
5. Apostrophe-like characters are normalised first (see APOSTROPHES):
   after o/g -> modifier (ў/ғ); elsewhere -> tutuq belgisi (ъ).
   NOTE: this normalisation is itself a typographic intervention; the paper's
   "latin_norm" control condition isolates it from the script change.
"""
from __future__ import annotations

import re
import unicodedata

# characters that all stand for the Uzbek modifier / tutuq in real-world text
APOSTROPHES = "'`´ʻʼ‘’ʹ′ʻʼ‘’"
_APOS_RE = re.compile("[" + re.escape(APOSTROPHES) + "]")
CANON_APOS = "ʻ"  # we normalise to U+02BB internally

LAT_VOWELS = set("aeiouAEIOU")
CYR_VOWELS = set("аеёиоуэюяўАЕЁИОУЭЮЯЎ")

# Russian-origin stems where Latin 'ts' corresponds to Cyrillic 'ц'
TS_STEMS = [
    "tsiya", "tsiy", "tsion", "tsent", "tsikl", "tsirk", "tsement", "tsiv",
    "militsiy", "polytsi", "sentsion",
]

# ---------------------------------------------------------------- Latin -> Cyrillic
_L2C_SINGLE = {
    "a": "а", "b": "б", "d": "д", "f": "ф", "g": "г", "h": "ҳ", "i": "и",
    "j": "ж", "k": "к", "l": "л", "m": "м", "n": "н", "o": "о", "p": "п",
    "q": "қ", "r": "р", "s": "с", "t": "т", "u": "у", "v": "в", "x": "х",
    "y": "й", "z": "з",
    # letters occasionally present in loanwords / proper names
    "c": "к", "w": "в",
}


def normalize_apostrophes(text: str, to: str = CANON_APOS) -> str:
    """Map every apostrophe-like character to a single canonical character."""
    return _APOS_RE.sub(to, text)


def _case_like(src: str, out: str) -> str:
    return out.upper() if src[:1].isupper() else out


def _safe_lower(t: str) -> str:
    """Per-character lowercase that never changes the string length
    (str.lower() can expand e.g. 'İ' to two characters and break index alignment)."""
    return "".join(c.lower() if len(c.lower()) == 1 else c for c in t)


def lat2cyr(text: str) -> str:
    """Latin Uzbek -> Cyrillic Uzbek (see module docstring for limitations)."""
    t = normalize_apostrophes(unicodedata.normalize("NFC", text))
    out: list[str] = []
    n = len(t)
    i = 0
    prev_is_vowel = False  # previous *Latin* char was a vowel letter
    at_word_start = True
    low = _safe_lower(t)

    while i < n:
        ch = t[i]
        lc = low[i]
        nxt = low[i + 1] if i + 1 < n else ""
        nxt2 = low[i + 2] if i + 2 < n else ""

        if not ch.isalpha() and ch != CANON_APOS:
            out.append(ch)
            at_word_start = True
            prev_is_vowel = False
            i += 1
            continue

        # ---- ts-stems (Russian loans)
        if lc == "t" and nxt == "s":
            rest = low[i:i + 8]
            if any(rest.startswith(stem) for stem in TS_STEMS):
                out.append(_case_like(ch, "ц"))
                i += 2
                at_word_start = False
                prev_is_vowel = False
                continue

        # ---- modifier letters oʻ, gʻ
        if lc in "og" and nxt == CANON_APOS:
            out.append(_case_like(ch, "ў" if lc == "o" else "ғ"))
            i += 2
            at_word_start = False
            prev_is_vowel = (lc == "o")
            continue

        # ---- digraphs sh, ch, ng
        if lc == "s" and nxt == "h":
            out.append(_case_like(ch, "ш")); i += 2
            at_word_start = prev_is_vowel = False; continue
        if lc == "c" and nxt == "h":
            out.append(_case_like(ch, "ч")); i += 2
            at_word_start = prev_is_vowel = False; continue
        if lc == "n" and nxt == "g":
            # 'ng' -> 'нг' (two Cyrillic letters)
            if ch.isupper() and t[i + 1].isupper():
                out.append("НГ")
            elif ch.isupper():
                out.append("Нг")
            else:
                out.append("нг")
            i += 2
            at_word_start = prev_is_vowel = False; continue

        # ---- y + vowel
        if lc == "y":
            if nxt == "e":
                out.append(_case_like(ch, "е")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "o" and nxt2 != CANON_APOS:
                out.append(_case_like(ch, "ё")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "u":
                out.append(_case_like(ch, "ю")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "a":
                out.append(_case_like(ch, "я")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            out.append(_case_like(ch, "й")); i += 1
            at_word_start = False; prev_is_vowel = False; continue

        # ---- e
        if lc == "e":
            out.append(_case_like(ch, "э" if (at_word_start or prev_is_vowel) else "е"))
            i += 1
            at_word_start = False; prev_is_vowel = True; continue

        # ---- standalone apostrophe -> tutuq belgisi
        if ch == CANON_APOS:
            out.append("ъ"); i += 1
            at_word_start = False; prev_is_vowel = False; continue

        # ---- plain letters
        if lc in _L2C_SINGLE:
            out.append(_case_like(ch, _L2C_SINGLE[lc]))
            prev_is_vowel = lc in "aiou"
        else:
            out.append(ch)  # already Cyrillic or other script: keep
            prev_is_vowel = ch in CYR_VOWELS
        at_word_start = False
        i += 1

    return "".join(out)


# ---------------------------------------------------------------- Cyrillic -> Latin
_C2L_SINGLE = {
    "а": "a", "б": "b", "в": "v", "г": "g", "д": "d", "ж": "j", "з": "z",
    "и": "i", "й": "y", "к": "k", "л": "l", "м": "m", "н": "n", "о": "o",
    "п": "p", "р": "r", "с": "s", "т": "t", "у": "u", "ф": "f", "х": "x",
    "ч": "ch", "ш": "sh", "э": "e", "ю": "yu", "я": "ya", "ё": "yo",
    "ў": "o" + CANON_APOS, "қ": "q", "ғ": "g" + CANON_APOS, "ҳ": "h",
    "ъ": CANON_APOS, "ь": "", "ы": "i", "ц": "s", "щ": "shch",
}


def cyr2lat(text: str) -> str:
    out: list[str] = []
    prev_cyr_vowel = False
    at_word_start = True
    for ch in text:
        lc = ch.lower()
        if lc == "е":  # Cyrillic Ie
            r = "ye" if (at_word_start or prev_cyr_vowel) else "e"
            out.append(r.capitalize() if ch.isupper() else r)
        elif lc in _C2L_SINGLE:
            r = _C2L_SINGLE[lc]
            out.append(r.capitalize() if (ch.isupper() and r) else r)
        else:
            out.append(ch)
        if ch.isalpha():
            at_word_start = False
            prev_cyr_vowel = lc in CYR_VOWELS
        else:
            at_word_start = True
            prev_cyr_vowel = False
    return "".join(out)


# ---------------------------------------------------------------- helpers for the experiments
def latin_normalised(text: str) -> str:
    """CONTROL condition: Latin text with apostrophes canonicalised to ASCII "'".
    Isolates the effect of typographic normalisation from the script change."""
    return normalize_apostrophes(text, to="'")


def script_share(text: str) -> dict:
    """Return fraction of alphabetic characters per script (Latin / Cyrillic / other)."""
    lat = cyr = oth = 0
    for ch in text:
        if not ch.isalpha():
            continue
        name = unicodedata.name(ch, "")
        if name.startswith("LATIN"):
            lat += 1
        elif name.startswith("CYRILLIC"):
            cyr += 1
        else:
            oth += 1
    tot = max(lat + cyr + oth, 1)
    return {"latin": lat / tot, "cyrillic": cyr / tot, "other": oth / tot}


def char_error_rate(ref: str, hyp: str) -> float:
    """Levenshtein CER (no external deps)."""
    if not ref:
        return 0.0 if not hyp else 1.0
    prev = list(range(len(hyp) + 1))
    for i, rc in enumerate(ref, 1):
        cur = [i]
        for j, hc in enumerate(hyp, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rc != hc)))
        prev = cur
    return prev[-1] / len(ref)


if __name__ == "__main__":
    demo = "Oʻzbekiston Respublikasi, gʻarbda Yevropa; yoʻq, ma'no, ekran, ongli, Toshkent, informatsiya."
    print(demo)
    print(lat2cyr(demo))
    print(cyr2lat(lat2cyr(demo)))


In [ ]:
%%writefile validate_feruza.py
"""
validate_feruza.py -- aggregate-only validation of lat2cyr against FeruzaSpeech transcripts.

Terms of the dataset: academic/non-commercial use, no redistribution of text -> this script prints
ONLY aggregate statistics and character-level confusion counts (single characters), never sentences/words.
Needs an HF token with granted access (Kaggle secret HF_TOKEN).
usage: python validate_feruza.py <hf_dataset_id> [max_rows]
"""
import re, sys, difflib, statistics
from collections import Counter
from translit_uz import lat2cyr, char_error_rate

DS = sys.argv[1]
MAXR = int(sys.argv[2]) if len(sys.argv) > 2 else 4000   # rows per split
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import os; tok = os.environ.get("HF_TOKEN")
from datasets import load_dataset
from itertools import islice
ds = load_dataset(DS, token=tok, streaming=True)   # streaming: avoids downloading the audio of the whole corpus
print("splits:", list(ds.keys()))
split0 = list(ds.keys())[0]
print("features:", {k: str(v)[:60] for k, v in ds[split0].features.items()})
CYR = re.compile(r"[\u0400-\u04FF]"); LAT = re.compile(r"[A-Za-z]")

def kind(s):
    c, l = len(CYR.findall(s)), len(LAT.findall(s))
    return "cyr" if c > 2 * l else "lat" if l > 2 * c else "mixed"

head = list(islice(ds[split0], 200))
cols = [c for c in head[0] if isinstance(head[0][c], str)]
votes = {c: Counter(kind(r[c]) for r in head if isinstance(r[c], str)) for c in cols}
print("string columns and script votes:", {c: dict(v) for c, v in votes.items()})
cyr_col = max(cols, key=lambda c: votes[c]["cyr"], default=None)
lat_col = max(cols, key=lambda c: votes[c]["lat"], default=None)
print("using Latin column:", lat_col, "| Cyrillic column:", cyr_col)
if not cyr_col or not lat_col or votes[cyr_col]["cyr"] == 0 or votes[lat_col]["lat"] == 0:
    sys.exit("Could not find both a Latin and a Cyrillic text column; inspect the printed columns.")

for sp in ds.keys():
    d = ds[sp]
    cers, exact, wok, wall, n = [], 0, 0, 0, 0
    conf = Counter()
    for i, r in enumerate(d):
        if i >= MAXR: break
        lat, cyr = r.get(lat_col), r.get(cyr_col)
        if not lat or not cyr: continue
        h = lat2cyr(lat); n += 1
        cers.append(char_error_rate(cyr, h)); exact += h == cyr
        hw, rw = h.split(), cyr.split()
        if len(hw) == len(rw):
            wall += len(rw); wok += sum(a == b for a, b in zip(hw, rw))
        sm = difflib.SequenceMatcher(None, cyr, h, autojunk=False)
        for op, a0, a1, b0, b1 in sm.get_opcodes():
            if op == "replace" and (a1 - a0) == (b1 - b0):
                for x, y in zip(cyr[a0:a1], h[b0:b1]): conf[(x, y)] += 1
            elif op == "delete":   # present in human, missing in hypothesis
                for x in cyr[a0:a1]: conf[(x, "<none>")] += 1
            elif op == "insert":   # extra in hypothesis
                for y in h[b0:b1]: conf[("<none>", y)] += 1
    if not n: continue
    q = sorted(cers)
    print(f"\n[{sp}] n={n} meanCER={sum(cers)/n:.4f} medianCER={statistics.median(cers):.4f} "
          f"p90={q[int(.9*(n-1))]:.4f} sentExact={exact/n:.3f} wordExact={wok/max(wall,1):.3f} (over {wall} words)")
    print("top single-character confusions (human -> rule-based): count")
    for (x, y), c in conf.most_common(15):
        print(f"  {x!r} -> {y!r}: {c}")


In [ ]:
# Kaggle: Add-ons > Secrets: add HF_TOKEN (read token of the Hugging Face account that has access) and attach it
!python validate_feruza.py k2speech/FeruzaSpeech_parquet_dataset